# 02 · QAOA vs classical baselines

The full sweep lives in `src/microgrid/benchmark.py`. Reproduce it from the repository root with:

```
python -m microgrid.benchmark --n 6 8 10 12 --density 0.2 0.4 0.6 --seeds 0 1 2 --max-p 3 --noisy
```

This notebook first walks through one instance step by step, then loads the saved sweep from `results/`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import matplotlib.pyplot as plt

## One instance, layer by layer

In [ ]:
from microgrid import make_microgrid, build_qubo, prepare, brute_force, solve_layerwise, probability_of_optimal

grid = make_microgrid(n=10, density=0.3, seed=1)
problem = prepare(build_qubo(grid))
exact, land = brute_force(problem.qubo)
p_random = len(land.optimal_indices) / 2**grid.n

results = solve_layerwise(problem, max_reps=3)
for r in results:
    p_opt = probability_of_optimal(r.probs, land.optimal_indices)
    print(f"p={r.reps}  approx ratio of <E> = {land.approximation_ratio(r.expectation):.3f}  "
          f"P(optimal) = {p_opt:.4f}  ({p_opt / p_random:.0f}x random)  evaluations = {r.nfev}")

In [ ]:
from microgrid.qaoa import top_bitstrings

top = top_bitstrings(results[-1].probs, grid.n, 12)
colors = ["#2E9E5B" if np.isclose(problem.energies[int(b, 2)], exact.energy) else "#9AA5B1" for b, _ in top]
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(len(top)), [p for _, p in top], color=colors)
ax.set_xticks(range(len(top)))
ax.set_xticklabels([b for b, _ in top], rotation=60, family="monospace", fontsize=8)
ax.set_ylabel("probability")
ax.set_title("Most likely dispatches after QAOA (green = optimal)")
plt.show()

## CVaR objective

Optimising the mean of the best 25% of outcomes instead of the full mean (Barkoutsos et al., Quantum 2020) focuses the optimiser on what we keep: the best sample.

In [ ]:
cvar_results = solve_layerwise(problem, max_reps=3, objective="cvar", alpha=0.25)
for r_mean, r_cvar in zip(results, cvar_results):
    print(f"p={r_mean.reps}  P(optimal): mean objective {probability_of_optimal(r_mean.probs, land.optimal_indices):.4f}"
          f"  |  CVaR objective {probability_of_optimal(r_cvar.probs, land.optimal_indices):.4f}")

## Saved sweep

In [ ]:
from IPython.display import Image, Markdown, display

results_dir = ROOT / "results"
summary = results_dir / "summary.md"
if summary.exists():
    display(Markdown(summary.read_text(encoding="utf-8")))
    for png in sorted((results_dir / "figures").glob("*.png")):
        display(Image(filename=str(png)))
else:
    print("Run the benchmark command above first.")